## 说明
代码演示的功能是 RAG (Retrieval-Augmented Generation，检索增强生成)，即 基于文档的问答系统。

简短说明
该功能的核心目标是：

检索（Retrieval）： 将一份外部文档（例如您的 OutdoorClothingCatalog_1000.csv 产品目录）加载到 向量数据库 中，以便进行语义搜索。

增强（Augmented）： 当用户提出问题时，系统会先从向量数据库中检索出与问题最相关的几段文本。

生成（Generation）： 将用户的 原始问题 和检索到的 相关上下文 一起喂给大型语言模型（LLM），由 LLM 结合上下文来生成精确的答案。

简而言之，它创建了一个能够精确回答用户关于特定文档（如产品目录）内容的问答系统，从而克服了 LLM 知识的局限性。

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# --- 1. 环境准备 (与旧版文件保持一致) ---
# 确保加载环境变量
_ = load_dotenv(find_dotenv()) 


# --- 2. 导入更新 (使用模块化的新包) ---
from langchain_openai import ChatOpenAI  # 导入模型和嵌入
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader    # 导入文档加载器
from langchain_community.vectorstores import DocArrayInMemorySearch # 导入内存向量存储
from langchain_core.prompts import ChatPromptTemplate       # 导入核心 Prompt
from langchain_core.runnables import RunnablePassthrough    # 导入 LCEL 核心组件
from langchain_core.output_parsers import StrOutputParser   # 导入输出解析器
from IPython.display import display, Markdown

# 定义文件和查询
file = 'OutdoorClothingCatalog_1000.csv'
# 翻译：请用 Markdown 格式的表格列出所有具有防晒功能的衬衫，并对每件衬衫进行简要描述。
query = "Please list all your shirts with sun protection in a table in markdown and summarize each one."

# --- 3. 数据加载与嵌入 (取代 VectorstoreIndexCreator) ---

# 3.1. 加载数据
loader = CSVLoader(
    file_path=file,
    # 强制指定编码为 UTF-8，解决 UnicodeDecodeError
    encoding='utf-8' 
)
docs = loader.load()
# docs[0] 包含了一条产品记录

# 3.2. 初始化嵌入模型
# model_name：指定要加载的模型名称 (例如 BGE-Small)
# model_kwargs：配置模型加载参数，如 device="cpu" 或 "cuda"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"}  # 注意：参数名统一为小写
)

# 3.3. 创建向量数据库 (取代 VectorstoreIndexCreator.from_loaders)
# 直接从 docs 和 embeddings 创建内存向量数据库
db = DocArrayInMemorySearch.from_documents(
    docs, 
    embeddings
)

# 3.4. 获取检索器
retriever = db.as_retriever()
# 检索器是 RAG 链的核心组件之一

# --- 4. RAG 链的 LCEL 实现 (取代 RetrievalQA) ---

# 4.1. 初始化 LLM
llm = ChatOpenAI(
    temperature=0.0, 
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
    )

# 4.2. 定义 RAG Prompt 模板
# RAG Prompt 必须接收 {context} 和 {question} 两个变量
RAG_PROMPT_TEMPLATE = """
You are a professional outdoor apparel catalog assistant.
Use the following pieces of retrieved context to answer the question. 
If you cannot find the answer in the context, politely state that you cannot answer based on the provided information.

Context:
{context}

Question: {question}
"""
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

# 4.3. 组织 LCEL RAG 链
# 链的输入必须是 {"question": query_string} 字典格式

def format_docs(docs):
    """
    一个辅助函数，将检索到的文档列表格式化为一个大的字符串，供 LLM 使用。
    """
    texts = []
    for doc in docs:
        content = doc.page_content
        if not isinstance(content, str):
            content = str(content)
        texts.append(content)
    return "\n\n".join(texts)

# LCEL RAG 链的核心结构：
rag_chain = (
    # 步骤 1: 定义 Chain 的输入结构 (字典)
    {
        # 'context' 键的值通过管道连接到 retriever，
        # retriever 接收 'question' 键的值进行检索，并将结果传给 format_docs
        "context": retriever | format_docs, 
        
        # 'question' 键的值直接通过，作为后续 Prompt 的输入
        "question": RunnablePassthrough() 
    }
    # 步骤 2: 将格式化后的字典传入 Prompt
    | rag_prompt
    # 步骤 3: 调用 LLM
    | llm
    # 步骤 4: 将 LLM 的 Message 输出解析为纯字符串
    | StrOutputParser()
)

print("===  RAG Chain 结构图 ===")
# .get_graph().draw_ascii() 作用：可视化 LCEL 链的结构，对于复杂 Chain 调试非常有帮助。
# 结构图将展示：Routing Chain -> RunnableBranch [Physics Path, Math Path, Default Path]
print(rag_chain.get_graph().print_ascii())

# --- 5. 执行调用 ---
print(f"Executing RAG Chain with query: {query}")

# LCEL 链的调用方式：.invoke({})
response = rag_chain.invoke(query)

print("\n--- Final Response ---")
display(Markdown(response))

c:\Users\bangsun\miniconda3\envs\jupterlab\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


===  RAG Chain 结构图 ===
             +---------------------------------+          
             | Parallel<context,question>Input |          
             +---------------------------------+          
                    ***                ***                
                 ***                      ***             
               **                            ***          
+----------------------+                        **        
| VectorStoreRetriever |                         *        
+----------------------+                         *        
            *                                    *        
            *                                    *        
            *                                    *        
    +-------------+                       +-------------+ 
    | format_docs |                       | Passthrough | 
    +-------------+*                      +-------------+ 
                    ***                ***                
                       ***       

Certainly! Here is a table summarizing all the shirts with sun protection, along with a brief description of each:

| Shirt Name                  | Description                                                                                                                                                                                                                   | Size & Fit                         | Fabric & Care                                                                                      | Additional Features                                                                                                    |
|-----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------|---------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------|
| **Sunrise Tee**             | Stay cool, comfortable, and dry on the hottest days in this women's UV-protective button-down shirt. The lightweight, high-performance fabric wicks away moisture and dries quickly.                                           | Slightly Fitted: Softly shapes the body. Falls at hip. | 71% nylon, 29% polyester. Machine wash and dry.                                                   | Built-in SunSmart™ UPF 50+ rated, wrinkle-free, smoother buttons, low-profile pockets, side shaping, cape venting.      |
| **Women's Tropical Tee, Sleeveless** | A five-star sleeveless button-up shirt with a fit to flatter and SunSmart™ protection to block the sun’s harmful UV rays.                                                                                                       | Slightly Fitted: Softly shapes the body. Falls at hip. | Shell: 71% nylon, 29% polyester. Cape lining: 100% polyester. Machine wash and dry.               | Built-in SunSmart™ UPF 50+ rated, updated design, smoother buttons, wrinkle resistant, low-profile pockets, side shaping, cape venting. |
| **Sun Shield Shirt**        | Block the sun, not the fun – this high-performance sun shirt is guaranteed to protect from harmful UV rays.                                                                                                                    | Slightly Fitted: Softly shapes the body. Falls at hip. | 78% nylon, 22% Lycra Xtra Life fiber. Handwash, line dry.                                        | UPF 50+ rated, wicks moisture, quick-drying comfort, fits over swimsuits, abrasion resistant, recommended by The Skin Cancer Foundation. |
| **Men's Tropical Plaid Short-Sleeve Shirt** | Our lightest hot-weather shirt is rated UPF 50+ for superior protection from the sun's UV rays. With a traditional fit that is relaxed through the chest, sleeve, and waist, this fabric is made of 100% polyester and is wrinkle-resistant. | Traditional fit: Relaxed through the chest, sleeve, and waist. | 100% polyester. Imported.                                                                         | Front and back cape venting, two front bellows pockets, highest rated sun protection possible.                          |

### Summary:
- **Sunrise Tee**: A women's UV-protective button-down shirt with built-in SunSmart™ UPF 50+ protection, moisture-wicking, and quick-drying properties. It features a slightly fitted design, wrinkle-free fabric, and various functional details like smoother buttons and low-profile pockets.
- **Women's Tropical Tee, Sleeveless**: A sleeveless button-up shirt designed for women, offering SunSmart™ UPF 50+ protection. It has a slightly fitted design, wrinkle-resistant fabric, and includes features like smoother buttons and low-profile pockets for a flattering fit.
- **Sun Shield Shirt**: A high-performance sun shirt that provides UPF 50+ protection. It is made of a blend of nylon and Lycra Xtra Life fiber, making it moisture-wicking and quick-drying. This shirt is also abrasion-resistant and recommended by The Skin Cancer Foundation.
- **Men's Tropical Plaid Short-Sleeve Shirt**: A men's short-sleeve shirt with UPF 50+ sun protection. It has a traditional, relaxed fit and is made of 100% polyester, which is wrinkle-resistant. The shirt includes front and back cape venting and two front bellows pockets for added functionality.